# TOOL Notebook 2 — Preprocessing & Dataset PyTorch
**ISIC Project | Segmentation de Lésions Cutanées**

Ce notebook couvre :
- OK Prétraitement des images (resize, normalisation)
- OK Augmentation de données
- OK Dataset et DataLoader PyTorch
- OK Vérification du pipeline

## LINK Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
from pathlib import Path

PROJECT_PATH  = '/content/drive/MyDrive/ISIC_Project'
IMAGES_PATH   = os.path.join(PROJECT_PATH, 'data', 'Images')
MASQUES_PATH  = os.path.join(PROJECT_PATH, 'data', 'Masques')
SRC_PATH      = os.path.join(PROJECT_PATH, 'src')
OUTPUTS_PATH  = os.path.join(PROJECT_PATH, 'outputs')
sys.path.append(SRC_PATH)
os.makedirs(OUTPUTS_PATH, exist_ok=True)

!pip install -q albumentations opencv-python-headless scikit-learn

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
import cv2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'OK Setup OK | Device : {device}')

## MAGNIFYING_GLASS 2.1 — Chargement des fichiers (filtre images uniquement)

In [ ]:
# OK CORRECTION : filtre les fichiers non-images (ATTRIBUTION.txt, etc.)
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

all_images = sorted([p for p in Path(IMAGES_PATH).glob('*.*')
                     if p.suffix.lower() in IMAGE_EXTENSIONS])
all_masks  = sorted([p for p in Path(MASQUES_PATH).glob('*.*')
                     if p.suffix.lower() in IMAGE_EXTENSIONS])

print(f'FRAME_WITH_PICTURE  Images  : {len(all_images)}')
print(f'MASK Masques : {len(all_masks)}')

assert len(all_images) == len(all_masks), \
    f'CROSS_MARK Nombre images ({len(all_images)}) != masques ({len(all_masks)})'
print('OK Correspondance OK !')

## TOOL 2.2 — Transforms & Augmentations

In [ ]:
IMG_SIZE = 256

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=30, p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.4),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

print(f'OK Train transforms : {len(train_transform.transforms)} étapes')
print(f'OK Val   transforms : {len(val_transform.transforms)} étapes')

## PACKAGE 2.3 — Classe ISICDataset

In [ ]:
class ISICDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img  = cv2.imread(str(self.image_paths[idx]))
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)

        if self.transform:
            out  = self.transform(image=img, mask=mask)
            img  = out['image']
            mask = out['mask'].unsqueeze(0)

        return img.float(), mask.float()

print('OK Classe ISICDataset définie !')

## ✂️ 2.4 — Split Train / Val / Test (70 / 15 / 15)

In [ ]:
train_imgs, temp_imgs, train_msks, temp_msks = train_test_split(
    all_images, all_masks, test_size=0.3, random_state=42)
val_imgs, test_imgs, val_msks, test_msks = train_test_split(
    temp_imgs, temp_msks, test_size=0.5, random_state=42)

print(f'REPORT Split des données :')
print(f'   Train : {len(train_imgs)} images ({len(train_imgs)/len(all_images)*100:.0f}%)')
print(f'   Val   : {len(val_imgs)} images ({len(val_imgs)/len(all_images)*100:.0f}%)')
print(f'   Test  : {len(test_imgs)} images ({len(test_imgs)/len(all_images)*100:.0f}%)')

## LAUNCH 2.5 — Création des DataLoaders

In [ ]:
BATCH_SIZE = 8

train_dataset = ISICDataset(train_imgs, train_msks, transform=train_transform)
val_dataset   = ISICDataset(val_imgs,   val_msks,   transform=val_transform)
test_dataset  = ISICDataset(test_imgs,  test_msks,  transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'OK DataLoaders créés !')
print(f'   Train : {len(train_loader)} batchs')
print(f'   Val   : {len(val_loader)} batchs')
print(f'   Test  : {len(test_loader)} batchs')

## 🔎 2.6 — Vérification du pipeline

In [ ]:
imgs, masks = next(iter(train_loader))
print(f'Batch image shape : {imgs.shape}   dtype: {imgs.dtype}')
print(f'Batch mask  shape : {masks.shape}  dtype: {masks.dtype}')
print(f'Image range : [{imgs.min():.2f}, {imgs.max():.2f}]')
print(f'Mask  range : [{masks.min():.2f}, {masks.max():.2f}]')

# Visualisation d'un batch augmenté
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    img_show = (imgs[i] * std + mean).clamp(0,1).permute(1,2,0).numpy()
    msk_show = masks[i, 0].numpy()
    axes[0,i].imshow(img_show);           axes[0,i].set_title(f'Image {i}');  axes[0,i].axis('off')
    axes[1,i].imshow(msk_show, cmap='gray'); axes[1,i].set_title(f'Masque {i}'); axes[1,i].axis('off')
plt.suptitle('Vérification pipeline — batch augmenté', fontsize=14)
plt.tight_layout()
plt.show()

## OK Résumé
- Fichiers non-images filtrés automatiquement
- Augmentations définies
- Split 70/15/15 effectué
- DataLoaders opérationnels

➡️ **Prochain notebook : `03_Modele_et_Entrainement.ipynb`**